# Split (test/train/val)

## 1. Librerias

In [47]:
import json
import gc
import numpy as np
import pandas as pd
import h5py
import s3fs
from sklearn.model_selection import train_test_split
import io
import boto3


## 2. Configuración general

In [ ]:
LOCAL_ENRICHED  = "./ml_dataset.parquet"
LOCAL_HDF5_IN   = "./deep_learning.h5"
 
LOCAL_TRAIN_ML  = "./ml_train.parquet"
LOCAL_VAL_ML    = "./ml_val.parquet"
LOCAL_TEST_ML   = "./ml_test.parquet"
LOCAL_SPLIT_IDS = "./split_ids.json"
 
S3_BASE = "s3://tfm-mu-bd/processed"
 
TRAIN_SIZE   = 0.70
VAL_SIZE     = 0.15
TEST_SIZE    = 0.15
RANDOM_STATE = 42
CHUNK_SIZE   = 500
 
s3 = s3fs.S3FileSystem()

## 3. Carga de datos

In [49]:
def load_enriched(path):
    df = pd.read_parquet(path)
    print(f"Registros totales: {len(df)}")
    print(f"Distribución is_adverse:\n{df['is_adverse'].value_counts()}\n")
    return df
 


## 4. Split estratificado

In [ ]:
def stratified_split(df):
    # Primer split: train (70%) vs temp (30%)
    df_train, df_temp = train_test_split(
        df,
        test_size=(VAL_SIZE + TEST_SIZE),
        stratify=df["is_adverse"],
        random_state=RANDOM_STATE
    )
 
    # Segundo split: val (15%) vs test (15%) sobre el temp
    val_relative = VAL_SIZE / (VAL_SIZE + TEST_SIZE)
    df_val, df_test = train_test_split(
        df_temp,
        test_size=(1 - val_relative),
        stratify=df_temp["is_adverse"],
        random_state=RANDOM_STATE
    )
 
    print("SPLIT RESULTANTE")
    for name, subset in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
        n   = len(subset)
        adv = subset["is_adverse"].sum()
        print(f"{name:5s}: {n:6d} registros | adversos: {adv:5d} ({adv/n*100:.1f}%)")
 
    return df_train, df_val, df_test
 

## 5. Guardar Split

In [51]:
def save_split_ids(df_train, df_val, df_test, path):
    split_ids = {
        "train": df_train["record_id"].tolist(),
        "val":   df_val["record_id"].tolist(),
        "test":  df_test["record_id"].tolist(),
    }
    with open(path, "w") as f:
        json.dump(split_ids, f, indent=2)
    print(f"\nSplit IDs guardados → {path}")
    return split_ids


In [52]:
def verify_splits(split_ids):
    train_ids = set(split_ids["train"])
    val_ids   = set(split_ids["val"])
    test_ids  = set(split_ids["test"])
 
    assert len(train_ids & val_ids)  == 0, "LEAKAGE: train ∩ val no vacío"
    assert len(train_ids & test_ids) == 0, "LEAKAGE: train ∩ test no vacío"
    assert len(val_ids   & test_ids) == 0, "LEAKAGE: val ∩ test no vacío"
 
    print("✓ Sin data leakage entre particiones")
    print(f"  train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")


In [53]:
def save_ml_splits(df_train, df_val, df_test):
    for df_part, local_path, s3_suffix in [
        (df_train, LOCAL_TRAIN_ML, "ml_train.parquet"),
        (df_val,   LOCAL_VAL_ML,   "ml_val.parquet"),
        (df_test,  LOCAL_TEST_ML,  "ml_test.parquet"),
    ]:
        df_part.to_parquet(local_path, index=False)
        s3.put(local_path, f"{S3_BASE}/machine_learning/{s3_suffix}")
        print(f"  ✓ {s3_suffix} ({len(df_part)} registros) → S3")
 
    print(f"ML Parquet guardado: train={len(df_train)} | val={len(df_val)} | test={len(df_test)}")


In [ ]:
def save_dl_splits_to_s3(hdf5_in_path, split_ids):
    s3_client = boto3.client("s3")
    BUCKET = "tfm-mu-bd"

    with h5py.File(hdf5_in_path, "r") as hf_in:
        all_ids = np.array([
            rid.decode() if isinstance(rid, bytes) else rid
            for rid in hf_in["record_id"][:]
        ])
        print(f"HDF5 total registros: {len(all_ids)}")

        for split_name in ["train", "val", "test"]:
            target_ids = set(split_ids[split_name])
            indices = np.where(np.isin(all_ids, list(target_ids)))[0]
            n = len(indices)
            s3_key = f"processed/deep_learning/dl_{split_name}.h5"
            print(f"Escribiendo {split_name}: {n} registros → s3://{BUCKET}/{s3_key}")

            # Escribir HDF5 en buffer de memoria
            buf = io.BytesIO()
            dt = h5py.string_dtype(encoding="utf-8")

            with h5py.File(buf, "w") as hf_out:
                ds_signal = hf_out.create_dataset("signal",
                                   shape=(n, 12, 5000), dtype="float32",
                                   compression="gzip", chunks=(min(64, n), 12, 5000))
                ds_age = hf_out.create_dataset("age", shape=(n,), dtype="float32")
                ds_sex = hf_out.create_dataset("sex", shape=(n,), dtype="int8")
                ds_record_id = hf_out.create_dataset("record_id", shape=(n,), dtype=dt)
                ds_dx = hf_out.create_dataset("dx",shape=(n,), dtype=dt)

                write_ptr = 0
                for chunk_start in range(0, len(indices), CHUNK_SIZE):
                    idx_chunk = sorted(indices[chunk_start:chunk_start + CHUNK_SIZE])
                    k = len(idx_chunk)
                    ds_signal[write_ptr:write_ptr+k] = hf_in["signal"][idx_chunk]
                    ds_age[write_ptr:write_ptr+k] = hf_in["age"][idx_chunk]
                    ds_sex[write_ptr:write_ptr+k] = hf_in["sex"][idx_chunk]
                    ds_record_id[write_ptr:write_ptr+k] = hf_in["record_id"][idx_chunk]
                    ds_dx[write_ptr:write_ptr+k] = hf_in["dx"][idx_chunk]
                    write_ptr += k
                    gc.collect()

            # Subir buffer a S3
            buf.seek(0)
            s3_client.upload_fileobj(buf, BUCKET, s3_key)
            del buf
            gc.collect()
            print(f"  dl_{split_name}.h5 listo en S3")
 


## 6. Verificación

In [ ]:
def verify_s3(split_ids):
    print("\n VERIFICACIÓN S3")
    expected = [
        f"{S3_BASE}/machine_learning/ml_train.parquet",
        f"{S3_BASE}/machine_learning/ml_val.parquet",
        f"{S3_BASE}/machine_learning/ml_test.parquet",
        f"{S3_BASE}/deep_learning/dl_train.h5",
        f"{S3_BASE}/deep_learning/dl_val.h5",
        f"{S3_BASE}/deep_learning/dl_test.h5",
        f"{S3_BASE}/metadata/split_ids.json",
    ]
    for path in expected:
        exists = s3.exists(path)
        status = "Existe" if exists else "No encontrado"
        size = f"({s3.info(path)['size'] / 1e6:.1f} MB)" if exists else ""
        print(f"  {status} {path} {size}")
 
    # Verificar proporciones en Parquet
    print("\nVERIFICACIÓN PROPORCIONES")
    for split_name in ["train", "val", "test"]:
        df = pd.read_parquet(f"{S3_BASE}/machine_learning/ml_{split_name}.parquet")
        n = len(df)
        adv = df["is_adverse"].sum()
        print(f"  {split_name:5s}: {n:6d} registros | adversos: {adv:5d} ({adv/n*100:.1f}%)")
        del df
        gc.collect()
 
 
if __name__ == "__main__":
 
    df = load_enriched(LOCAL_ENRICHED)
    df_train, df_val, df_test = stratified_split(df)
    del df
    gc.collect()
 
    split_ids = save_split_ids(df_train, df_val, df_test, LOCAL_SPLIT_IDS)
    verify_splits(split_ids)
 
    print("\n GUARDANDO ML (PARQUET)")
    save_ml_splits(df_train, df_val, df_test)
    del df_train, df_val, df_test
    gc.collect()
 
    s3.put(LOCAL_SPLIT_IDS, f"{S3_BASE}/metadata/split_ids.json")
    print("split_ids.json enviado a S3")
 
    print("\nGUARDANDO DL (HDF5)")
    save_dl_splits_to_s3(LOCAL_HDF5_IN, split_ids)
 
    verify_s3(split_ids)
 
    print("\nSPLIT COMPLETO")
